In [1]:
# IMPORT LIBRARY
import pandas as pd
import numpy as np

In [2]:
# MENGHITUNG VARIABEL HARGA OPSI PUT
df = pd.read_csv("Data_Kuotasi_Opsi_Put_NVDA_Cleaning.csv")
df["MID_PRICE"] = (df["P_BID"] + df["P_ASK"]) / 2

In [3]:
# MENGHITUNG LOG RETURN HARGA PENUTUPAN HARIAN SAHAM
df1 = pd.read_excel("Data_Harga_Penutupan_Harian_Saham.xlsx")
df1["QUOTE_DATE"] = pd.to_datetime(df1["QUOTE_DATE"])

df_return_saham = df1.copy()
df_return_saham["log_return"] = np.log(
    df1["UNDERLYING_LAST"] / 
    df1["UNDERLYING_LAST"].shift(1)
)
df_return_saham = df_return_saham.dropna(subset=["log_return"])

In [4]:
# MENYIMPAN DATA LOG RETURN
df_return_saham.to_excel("Data_Return_Harga_Saham.xlsx", index=False)

In [5]:
# MENGHITUNG VARIABEL VOLATILITAS (KONSTAN)
df_return_saham_bs = df_return_saham[
    (df_return_saham["QUOTE_DATE"] >= "2023-01-03") &
    (df_return_saham["QUOTE_DATE"] <= "2023-12-29")
]

df_return_saham_bs = df_return_saham_bs.dropna(subset=["log_return"])
std_return_saham = df_return_saham_bs["log_return"].std()
volatilitas_tahunan = std_return_saham * np.sqrt(252)

vol_konstan = df_return_saham_bs.copy()
vol_konstan["VOLATILITAS_KONSTAN"] = volatilitas_tahunan
columns_to_keep = [
    "QUOTE_DATE",
    "VOLATILITAS_KONSTAN"
]
vol_konstan = vol_konstan[columns_to_keep]
vol_konstan = vol_konstan.drop_duplicates(subset=['QUOTE_DATE']).reset_index(drop=True)

In [6]:
# MENGHITUNG VARIABEL VOLATILITAS (DINAMIS)
window = 90

vol_dinamis = df1.copy()
vol_dinamis["VOLATILITAS_DINAMIS"] = (
    df_return_saham["log_return"]
    .shift(1)
    .rolling(window=window)
    .apply(lambda x: np.sqrt(np.mean(x**2)) * np.sqrt(252), raw=True)
)

# Filter tanggal
vol_dinamis = vol_dinamis[
    (vol_dinamis["QUOTE_DATE"] >= "2022-04-01") &
    (vol_dinamis["QUOTE_DATE"] <= "2023-12-29")
]

columns_to_keep = [
    "QUOTE_DATE",
    "VOLATILITAS_DINAMIS"
]

vol_dinamis = vol_dinamis[columns_to_keep]
vol_dinamis = vol_dinamis.drop_duplicates(subset=['QUOTE_DATE']).reset_index(drop=True)

In [7]:
# MENGHITUNG VARIABEL TINGKAT SUKU BUNGA (KONSTAN)
df2 = pd.read_excel("Data_Tingkat_Suku_Bunga.xlsx")
df2["QUOTE_DATE"] = pd.to_datetime(df2["QUOTE_DATE"])

df["QUOTE_DATE"] = df["QUOTE_DATE"].astype(str).str.strip()
df["QUOTE_DATE"] = pd.to_datetime(df["QUOTE_DATE"], errors='coerce')
df_opsi = df.copy()

df_opsi = df_opsi.sort_values(["ID_OPTION", "QUOTE_DATE"]).reset_index(drop=True)
df2 = df2.sort_values("QUOTE_DATE").reset_index(drop=True)

df_dte_max = (
    df_opsi.groupby("ID_OPTION", as_index=False)["DTE"]
    .max()
    .rename(columns={"DTE": "DTE_MAX"})
)

df_opsi = df_opsi.merge(df_dte_max, on="ID_OPTION", how="left")
df_rate = df_opsi.merge(df2, on="QUOTE_DATE", how="left")

rate_cols = [
    "Treasury_1_Bulan",
    "Treasury_3_Bulan",
    "Treasury_6_Bulan",
    "Treasury_1_Tahun",
    "Treasury_2_Tahun"
]

df_rate[rate_cols] = df_rate[rate_cols].ffill()

tenor_points = np.array([1, 3, 6, 12, 24], dtype=float)
tenor_cols = [
    "Treasury_1_Bulan",
    "Treasury_3_Bulan",
    "Treasury_6_Bulan",
    "Treasury_1_Tahun",
    "Treasury_2_Tahun"
]

def interpolasi_suku_bunga(row):
    target_month = row["DTE_MAX"] / 30.0
    y = row[tenor_cols].values.astype(float)
    
    if target_month <= tenor_points[0]:
        return y[0]
        
    if target_month >= tenor_points[-1]:
        return y[-1]

    idx_upper = np.searchsorted(tenor_points, target_month)
    idx_lower = idx_upper - 1

    x0 = tenor_points[idx_lower]
    x1 = tenor_points[idx_upper]
    y0 = y[idx_lower]
    y1 = y[idx_upper]

    r_interp = y0 + ((target_month - x0) / (x1 - x0)) * (y1 - y0)
    return r_interp

df_rate["r_INTERP_HARIAN"] = df_rate.apply(interpolasi_suku_bunga, axis=1)

df_rf_konstan = (
    df_rate.groupby("ID_OPTION", as_index=False)["r_INTERP_HARIAN"]
    .mean()
    .rename(columns={"r_INTERP_HARIAN": "r_KONSTAN"})
)

rate_konstan = df_rate.merge(df_rf_konstan, on="ID_OPTION", how="left")
columns_to_keep = [
    "QUOTE_DATE",
    "ID_OPTION",
    "r_KONSTAN"
]
rate_konstan = rate_konstan[columns_to_keep].reset_index(drop=True)

In [8]:
# MENGHITUNG VARIABEL TINGKAT SUKU BUNGA (DINAMIS)
df["QUOTE_DATE"] = pd.to_datetime(df["QUOTE_DATE"])
df_merged = df.merge(df2, on="QUOTE_DATE", how="left")
df_merged = df_merged.sort_values("QUOTE_DATE")
df_merged = df_merged.ffill()

tenors = np.array([1/12, 3/12, 6/12, 1, 2])

def interpolate_rate(row):
    yields = row[["Treasury_1_Bulan","Treasury_3_Bulan","Treasury_6_Bulan","Treasury_1_Tahun","Treasury_2_Tahun"]].to_numpy(dtype=float)
    T = float(row["DTE"]) / 365
    return np.interp(T, tenors, yields)

rate_dinamis = df_merged.copy()
rate_dinamis["r_DINAMIS"] = df_merged.apply(interpolate_rate, axis=1)
columns_to_keep = [
    "QUOTE_DATE",
    "ID_OPTION",
    "r_DINAMIS"
]
rate_dinamis = rate_dinamis[columns_to_keep].reset_index(drop=True)

In [9]:
# DATA FINAL MODEL BLACK-SCHOLES
df_option_bs = df[
    (df['QUOTE_DATE'] >= "2023-01-03") &
    (df['QUOTE_DATE'] <= "2023-12-29")
]

df_merged_bs = df_option_bs.merge(vol_konstan, on="QUOTE_DATE", how="left")
df_merged_bs = df_merged_bs.sort_values("QUOTE_DATE")
df_merged_bs = df_merged_bs.ffill()

df_final_bs = df_merged_bs.merge(rate_konstan, on=["ID_OPTION", "QUOTE_DATE"], how="left")
df_final_bs = df_final_bs.sort_values("QUOTE_DATE")
df_final_bs = df_final_bs.ffill()

columns_to_keep = [
    "QUOTE_DATE",
    "ID_OPTION",
    "EXPIRE_DATE",
    "UNDERLYING_LAST",
    "STRIKE",
    "DTE",
    "VOLATILITAS_KONSTAN",
    "r_KONSTAN",
    "MID_PRICE"
]
df_final_bs = df_final_bs[columns_to_keep]
df_final_bs = df_final_bs.sort_values(["ID_OPTION", "QUOTE_DATE"]).reset_index(drop=True)

In [10]:
# MENYIMPAN DATA FINAL MODEL BLACK-SCHOLES 
df_final_bs.to_csv("Data_Final_Model_Black-Scholes.csv", index=False)

In [11]:
# DATA FINAL MODEL LSTM
df_merged_lstm = df.merge(vol_dinamis, on="QUOTE_DATE", how="left")
df_merged_lstm = df_merged_lstm.sort_values("QUOTE_DATE")
df_merged_lstm = df_merged_lstm.ffill()

df_final_lstm = df_merged_lstm.merge(rate_dinamis, on=["ID_OPTION", "QUOTE_DATE"], how="left")
df_final_lstm = df_final_lstm.sort_values("QUOTE_DATE")
df_final_lstm = df_final_lstm.ffill()

columns_to_keep = [
    "QUOTE_DATE",
    "ID_OPTION",
    "EXPIRE_DATE",
    "UNDERLYING_LAST",
    "STRIKE",
    "DTE",
    "VOLATILITAS_DINAMIS",
    "r_DINAMIS",
    "MID_PRICE"
]
df_final_lstm = df_final_lstm[columns_to_keep]
df_final_lstm = df_final_lstm.sort_values(["ID_OPTION", "QUOTE_DATE"]).reset_index(drop=True)

In [12]:
# MEYIMPAN DATA MODEL LSTM
df_final_lstm.to_csv("Data_Model_LSTM.csv", index=False)